###Silver Agents
🎯 What we do:
- clean agent data
- standardize fields
- remove duplicates

In [0]:
from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

In [0]:
agents_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_agents")
fraud_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_fraud_indicators")

claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims") \
    .select("claim_id") \
    .dropDuplicates()

In [0]:
agents_prepared = (
    agents_bronze
    .withColumn("agent_id", F.trim(F.col("agent_id")))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("bundesland", F.initcap(F.trim(F.col("bundesland"))))
    .withColumn("commission_rate", F.col("commission_rate").cast("double"))
    .withColumn("active_flag", F.col("active_flag").cast("boolean"))
)

In [0]:
silver_agents = (
    agents_prepared
    .filter(F.col("agent_id").isNotNull())
    .dropDuplicates(["agent_id"])
)

quarantine_agents = (
    agents_prepared
    .filter(F.col("agent_id").isNull())
    .withColumn("record_id", F.col("agent_id"))
    .withColumn("source_table", F.lit("bronze_agents"))
    .withColumn("error_reason", F.lit("invalid_agent_id"))
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

In [0]:
silver_agents.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_agents")

quarantine_agents.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_agents")

In [0]:
fraud_prepared = (
    fraud_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("previous_claims_count", F.col("previous_claims_count").cast("int"))
    .withColumn("suspicious_amount_flag", F.col("suspicious_amount_flag").cast("boolean"))
    .withColumn("duplicate_claim_flag", F.col("duplicate_claim_flag").cast("boolean"))
    .withColumn("late_report_flag", F.col("late_report_flag").cast("boolean"))
    .withColumn("high_risk_region_flag", F.col("high_risk_region_flag").cast("boolean"))
    .withColumn("risk_score", F.col("risk_score").cast("int"))
)

In [0]:
fraud_joined = (
    fraud_prepared.alias("f")
    .join(claims.alias("c"), "claim_id", "left")
    .withColumn("claim_exists", F.col("c.claim_id").isNotNull())
)

In [0]:
invalid_fraud = (
    fraud_joined
    .filter(
        F.col("claim_id").isNull()
        | (~F.col("claim_exists"))
        | (F.col("risk_score") < 0)
        | (F.col("risk_score") > 100)
    )
    .withColumn("record_id", F.col("claim_id"))
    .withColumn("source_table", F.lit("bronze_fraud_indicators"))
    .withColumn("error_reason", F.lit("fraud_validation_failed"))
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in fraud_prepared.columns]))
    )
)

In [0]:
silver_fraud = (
    fraud_joined
    .filter(
        F.col("claim_id").isNotNull()
        & F.col("claim_exists")
        & (F.col("risk_score").between(0, 100))
    )
    .drop("claim_exists")
    .dropDuplicates(["claim_id"])
)

In [0]:
silver_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators")

invalid_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_fraud_indicators")

In [0]:
print("Silver agents:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_agents").count())
print("Silver fraud:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators").count())

print("Quarantine fraud:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_fraud_indicators").count())

In [0]:
fraud_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_fraud_indicators")

claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims") \
    .select("claim_id") \
    .dropDuplicates()

fraud_clean = (
    fraud_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("previous_claims_count", F.col("previous_claims_count").cast("int"))
    .withColumn("suspicious_amount_flag", F.col("suspicious_amount_flag").cast("boolean"))
    .withColumn("duplicate_claim_flag", F.col("duplicate_claim_flag").cast("boolean"))
    .withColumn("late_report_flag", F.col("late_report_flag").cast("boolean"))
    .withColumn("high_risk_region_flag", F.col("high_risk_region_flag").cast("boolean"))
    .withColumn("risk_score", F.col("risk_score").cast("int"))
)

fraud_joined = fraud_clean.join(claims, "claim_id", "left") \
    .withColumn("claim_exists", F.col("claim_id").isNotNull())

silver_fraud = (
    fraud_joined
    .filter(
        (F.col("claim_exists")) &
        (F.col("risk_score").between(0, 100))
    )
    .dropDuplicates(["claim_id"])
)

silver_fraud.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators")